# 02 - Ingesta Bronze: RUES (Registro Mercantil)

Trabajo Práctico 1 - Sección 3 (Fuente 1): Pipeline de ingesta PySpark

## Fuente

* **Dataset**: Personas Naturales, Personas Jurídicas y Entidades Sin Ánimo de Lucro (RUES)
* **API**: `https://www.datos.gov.co/resource/c82u-588k.json` (Socrata / SoQL)
* **Volumen total en la fuente**: 9.407.309 registros (verificado con `$select=count(*)`)
* **Destino**: `Datos_Empresas.bronze.DE_Semiestructurado_RegistroMercantil_Api`

## Estrategia de carga: dos modos con el flag `MODO_DIARIO`

Este pipeline soporta dos formas de cargar RUES, controladas por una sola variable booleana:

* **`MODO_DIARIO = False`** (carga por mes): se cambian las variables `ANIO` y `MES` y se filtra por `fecha_actualizacion` para traer ese mes completo. La escritura usa `replaceWhere`: si el mes ya se había cargado antes, se trunca (borra) solo ese rango y se reemplaza con lo que traiga la API en esta ejecución; los demás meses ya cargados no se tocan.
* **`MODO_DIARIO = True`** (carga diaria incremental, igual patrón que TRM): se consulta `MAX(fecha_actualizacion)` ya cargado en la tabla, y se descarga solo lo que la API tenga **posterior a esa fecha** (`fecha_actualizacion > última_fecha_cargada`) hasta el momento actual. La escritura usa `mode("append")` — como nunca se vuelve a pedir un rango ya traído, no hace falta ningún chequeo extra de duplicados. Si la tabla todavía no existe, la primera carga se limita a los últimos `DIAS_CARGA_INICIAL` días (bootstrap).

En ambos casos el histórico ya cargado (meses u otras fechas anteriores) nunca se reprocesa ni se duplica.

## Regla de inmutabilidad (Capa Bronze)

* No se renombran columnas: se conservan exactamente los nombres que entrega la API (`codigo_camara`, `razon_social`, etc.).
* No se hace *casting* destructivo: todas las columnas de RUES son de tipo `text` en el origen, así que se dejan como texto (no se convierten fechas a `DATE` ni números a `INT`/`DOUBLE`). Esa limpieza es tarea de la futura Capa Silver.
* Solo se agregan las columnas de auditoría obligatorias: `_ingested_at` y `_source`.

In [ ]:
%python
import requests
import pandas as pd
from datetime import datetime, timedelta
from pyspark.sql import functions as F

BASE_URL = "https://www.datos.gov.co/resource/c82u-588k.json"
LIMIT = 50000  # tamaño de página soportado por Socrata
TABLA_DESTINO = "Datos_Empresas.bronze.DE_Semiestructurado_RegistroMercantil_Api"

# --- Flag principal: True = carga diaria incremental (watermark por fecha_actualizacion) ---
# ---              False = carga por mes (ANIO/MES, con replaceWhere)                     ---
MODO_DIARIO = False

# Solo se usan si MODO_DIARIO = False
ANIO = 2026
MES = 8

# Solo se usa si MODO_DIARIO = True y la tabla todavía no existe (primera carga)
DIAS_CARGA_INICIAL = 30


def rango_mes(anio, mes):
    inicio = datetime(anio, mes, 1)
    fin = datetime(anio + 1, 1, 1) if mes == 12 else datetime(anio, mes + 1, 1)
    return inicio.strftime("%Y/%m/%d"), fin.strftime("%Y/%m/%d")


def obtener_ultima_fecha_actualizacion(tabla):
    """Devuelve el fecha_actualizacion más reciente ya cargado en la tabla, o None si no existe."""
    if not spark.catalog.tableExists(tabla):
        return None
    fila = spark.table(tabla).agg(F.max("fecha_actualizacion").alias("max_fecha")).collect()[0]
    return fila["max_fecha"]


# fecha_actualizacion llega como texto 'YYYY/MM/DD HH:MM:SS...', comparable como string
if MODO_DIARIO:
    ultima_fecha = obtener_ultima_fecha_actualizacion(TABLA_DESTINO)

    if ultima_fecha is None:
        fecha_inicio = (datetime.now() - timedelta(days=DIAS_CARGA_INICIAL)).strftime("%Y/%m/%d")
        WHERE_CLAUSE = f"fecha_actualizacion >= '{fecha_inicio}'"
        print(f"Modo: DIARIO -- la tabla aún no existe, carga inicial desde {fecha_inicio} hasta ahora")
    else:
        WHERE_CLAUSE = f"fecha_actualizacion > '{ultima_fecha}'"
        print(f"Modo: DIARIO -- última fecha_actualizacion ya cargada: {ultima_fecha}")
        print("Se descarga todo lo nuevo desde esa fecha hasta ahora")
else:
    FECHA_INICIO, FECHA_FIN = rango_mes(ANIO, MES)
    WHERE_CLAUSE = f"fecha_actualizacion >= '{FECHA_INICIO}' AND fecha_actualizacion < '{FECHA_FIN}'"
    print(f"Modo: MENSUAL -> {ANIO}-{MES:02d} ({FECHA_INICIO} a {FECHA_FIN})")

print(f"Fuente: {BASE_URL}")
print(f"Filtro aplicado: {WHERE_CLAUSE}")

---

## Paso 1: Contar registros disponibles con el filtro aplicado

In [ ]:
%python
def contar_registros(where=None):
    params = {"$select": "count(*)"}
    if where:
        params["$where"] = where
    resp = requests.get(BASE_URL, params=params)
    resp.raise_for_status()
    return int(resp.json()[0]["count"])


total_registros = contar_registros(WHERE_CLAUSE)
print(f"Total de registros de RUES con el filtro aplicado: {total_registros:,}")

---

## Paso 2: Descargar los datos paginando (HTTP GET + `$limit`/`$offset`)

In [ ]:
%python
def descargar_datos(max_registros, where=None):
    registros = []
    offset = 0

    while offset < max_registros:
        pagina_limit = min(LIMIT, max_registros - offset)
        params = {"$limit": pagina_limit, "$offset": offset}
        if where:
            params["$where"] = where
        resp = requests.get(BASE_URL, params=params)
        resp.raise_for_status()
        pagina = resp.json()

        if not pagina:
            break

        registros.extend(pagina)
        offset += LIMIT
        print(f"Descargados {len(registros)} de {max_registros} registros...")

    return registros


if total_registros > 0:
    registros = descargar_datos(total_registros, WHERE_CLAUSE)
    df_pandas = pd.DataFrame(registros)
    print(f"\nDataset descargado: {df_pandas.shape[0]} filas x {df_pandas.shape[1]} columnas")
    df_pandas.head()
else:
    df_pandas = None
    print("No hay registros nuevos que cargar con el filtro aplicado.")

---

## Paso 3: Convertir a Spark DataFrame (sin transformar tipos) y agregar columnas de auditoría

In [ ]:
%python
# Se respeta el tipo con el que Socrata entrega cada campo (todo texto en este dataset).
# No se hace ningún .astype()/cast manual: eso sería un casteo destructivo, prohibido en Bronze.
if df_pandas is not None:
    df_spark = spark.createDataFrame(df_pandas)

    df_bronze = (
        df_spark
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source", F.lit(BASE_URL))
    )

    df_bronze.printSchema()
    display(df_bronze.limit(10))
else:
    df_bronze = None

---

## Paso 4: Persistir en Delta Lake

* **Modo mensual** (`MODO_DIARIO = False`): si la tabla no existe todavía, se crea con el primer mes. Si ya existe, se usa `replaceWhere` con el mismo filtro de fecha del mes actual: eso borra e inserta de nuevo *solo* ese rango, dejando intactos los meses cargados en ejecuciones anteriores.
* **Modo diario** (`MODO_DIARIO = True`): como el filtro ya pidió a la API solo lo posterior a `MAX(fecha_actualizacion)` ya cargado, todo lo que llega aquí es nuevo por construcción — se escribe directo con `mode("append")`, sin necesidad de deduplicar.

In [ ]:
%python
if df_bronze is None:
    print("Nada que escribir en esta ejecución (0 registros nuevos).")
elif MODO_DIARIO:
    (
        df_bronze.write
        .format("delta")
        .mode("append")
        .saveAsTable(TABLA_DESTINO)
    )
    print(f"Modo diario: se agregaron {df_bronze.count()} registros nuevos a {TABLA_DESTINO} (append, sin reprocesar histórico).")
else:
    tabla_existe = spark.catalog.tableExists(TABLA_DESTINO)
    writer = df_bronze.write.format("delta").mode("overwrite")

    if tabla_existe:
        # Trunca solo el rango de fecha_actualizacion del mes actual; no toca otros meses ya cargados
        writer = writer.option("replaceWhere", WHERE_CLAUSE)
        print(f"Tabla existente: se trunca y recarga solo el mes {ANIO}-{MES:02d}")
    else:
        writer = writer.option("overwriteSchema", "true")
        print("Tabla nueva: se crea con este primer mes cargado")

    writer.saveAsTable(TABLA_DESTINO)
    print(f"Tabla Delta actualizada: {TABLA_DESTINO} (mes {ANIO}-{MES:02d})")

In [0]:
%sql
-- Verificación rápida de la ingesta
DESCRIBE EXTENDED Datos_Empresas.bronze.DE_Semiestructurado_RegistroMercantil_Api;

In [0]:
%sql
SELECT COUNT(*) AS total_filas, MIN(_ingested_at) AS primera_carga, MAX(_ingested_at) AS ultima_carga
FROM Datos_Empresas.bronze.DE_Semiestructurado_RegistroMercantil_Api;


In [0]:
%sql
-- Ver cuántas filas hay cargadas por cada mes (fecha_actualizacion) para confirmar
-- que se van acumulando meses sin duplicar
SELECT LEFT(fecha_actualizacion, 7) AS mes, COUNT(*) AS filas
FROM Datos_Empresas.bronze.DE_Semiestructurado_RegistroMercantil_Api
GROUP BY LEFT(fecha_actualizacion, 7)
ORDER BY mes;

In [0]:
%sql
SELECT * FROM Datos_Empresas.bronze.DE_Semiestructurado_RegistroMercantil_Api LIMIT 10;

---

## Siguiente paso

Continuar con [`03_Ingesta_Bronze_TRM.ipynb`](03_Ingesta_Bronze_TRM.ipynb) para la fuente complementaria con carga incremental.